### 📘 SLM 기반 멀티모달 에이전트 구현 (LangChain + Ollama)

이 코드는 로컬 환경에서 **Ollama**와 **LangChain**을 활용하여 이미지를 보고 분석할 수 있는 **멀티모달 에이전트(Multimodal Agent)** 를 구축하는 예제입니다.

**주요 기능 및 절차:**

1.  **환경 설정 (`Cell 1`)**:
    * 로컬 LLM 구동을 위한 **Ollama** 설치 및 서버 실행
    * LangChain, LangGraph, Ollama 연동을 위한 필수 Python 라이브러리 설치
2.  **에이전트 구현 (`Cell 2`)**:
    * **모델**: 시각 정보 처리가 가능한 `qwen3.5:4b` 모델 사용
    * **도구(Tool)**: 분석된 내용을 파일(`report.txt`)로 저장하는 `save_file_tool` 정의
    * **실행**: 이미지를 입력받아 내용을 요약하고, 에이전트가 스스로 도구를 호출하여 결과를 파일로 저장

> **💡 중요**: 첫 번째 셀(라이브러리 설치) 실행 후 `ModuleNotFoundError`가 발생한다면, 상단 메뉴의 **[런타임] > [세션 다시 시작]** 을 클릭하여 커널을 재시작한 뒤 두 번째 셀을 실행해주세요.

### 라이브러리 버전 문제로 세션이 다시 시작될 수 있습니다.

In [1]:
# # 라이브러리 설치 (LangChain 1.0 + LangGraph Engine)
# import os
# import shutil
# import sys

# # 1. Ollama 설치
# print("🔄 Ollama 설치 확인 중...")
# if shutil.which("ollama") is None:
#     os.system("curl -fsSL https://ollama.com/install.sh | sh")
#     os.system("sudo apt-get install -y zstd")

# # 2. Python 라이브러리 설치
# # 조건문 없이 강제로 최신 버전을 설치하거나, 필요한 패키지를 구체적으로 명시합니다.
# print("🐍 Python 필수 라이브러리 설치 중...")

# # Colab 등의 노트북 환경에서는 !pip (매직 커맨드) 사용 권장
# # -U 옵션으로 항상 최신 버전으로 업데이트하여 호환성 문제 방지
# !pip install -U langchain langchain-ollama langchain-community langgraph pillow

# print("\n🎉 환경 준비 완료! 상단 메뉴의 [런타임] > [세션 다시 시작]을 실행해주세요.")

### ⚙️ 1. 모듈 임포트 및 기본 설정
* 에이전트 구현에 필요한 LangChain 및 이미지 처리 모듈을 임포트합니다.
* 사용할 모델(`qwen3.5:4b`)과 분석할 이미지의 경로를 설정합니다.

In [2]:
1# LangChain 1.0 Agent 구현
import os
import time
import subprocess
import base64
from PIL import Image, ImageDraw

# --- [수정 1] 올바른 임포트 경로 ---
from langchain_ollama import ChatOllama
# tool은 langchain_core.tools에서 가져와야 합니다.
from langchain_core.tools import tool
from langchain.agents import create_agent  # New import
from langchain_core.messages import HumanMessage, SystemMessage

# ==========================================
# [설정] 모델 및 경로
# ==========================================
MODEL_NAME = "qwen3.5:4b"
# ministral-3:3b
# ministral-3:8b
# qwen3.5:2b
# qwen3.5:4b
# 1. VRAM에 맞는지
# 2. vision 능력 갖췄는지
# 3. tool calling 능력 갖췄는지



IMAGE_PATH = "/home/user/MultiAgents/Day_2/Ex/image_data/cat.jpg"


### 🚀 2. Ollama 서버 실행 및 모델 다운로드
* 로컬에서 Ollama 서버가 실행 중인지 확인하고 시작합니다.
* 설정한 모델(`qwen3.5:4b`)이 로컬에 없다면 자동으로 다운로드하여 준비 상태를 만듭니다.

In [3]:
# ==========================================
# 1. Ollama 서버 및 모델 준비
# ==========================================

# Ollama 서버를 실행하고 지정한 모델이 준비되어 있는지 확인하는 함수를 정의합니다.
def start_ollama_server():
    # 현재 Ollama 서버 프로세스가 실행 중인지 확인합니다.
    if os.system("pgrep -x ollama > /dev/null") != 0:
        # Ollama 서버가 실행 중이 아니면 시작 메시지를 출력합니다.
        print("🚀 Ollama 서버 시작...")
        # Ollama 서버를 백그라운드에서 실행하고 출력은 숨깁니다.
        subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        # 서버가 뜰 시간을 주기 위해 5초 대기합니다.
        time.sleep(5)

    # 현재 로컬에 설치된 Ollama 모델 목록을 가져옵니다.
    result = subprocess.run(["ollama", "list"], capture_output=True, text=True)
    # 지정한 모델이 목록에 없으면 다운로드를 진행합니다.
    if MODEL_NAME not in result.stdout:
        # 다운로드할 모델 이름을 안내합니다.
        print(f"📥 모델 다운로드 중: {MODEL_NAME}...")
        # Ollama에서 해당 모델을 내려받습니다.
        subprocess.run(["ollama", "pull", MODEL_NAME], check=True)
    # 서버와 모델 준비가 완료되었음을 출력합니다.
    print(f"✅ 모델 준비됨: {MODEL_NAME}")

### 🔧 3. 이미지 처리 유틸리티 및 도구(Tool) 정의
* 이미지를 모델이 이해할 수 있는 Base64 포맷으로 인코딩하는 함수와 에이전트가 분석 결과를 텍스트 파일로 저장할 때 사용할 커스텀 도구(`save_file_tool`)를 정의합니다.

In [4]:
# ==========================================
# 2. 유틸리티 및 도구(Tool) 정의
# ==========================================


# 웹 상의 이미지도 아래 로직으로 핸들링 가능합니다
def encode_image(image_source):
    # HTTP URL이면 fetch해서 base64 변환
    if image_source.startswith("http://") or image_source.startswith("https://"):
        return base64.b64encode(requests.get(image_source).content).decode("utf-8")
    # 로컬 파일이면 기존 로직
    try:
        with open(image_source, "rb") as image_file:
            return base64.b64encode(image_file.read()).decode("utf-8")
    except FileNotFoundError:
        create_dummy_image(image_source)
        with open(image_source, "rb") as image_file:
            return base64.b64encode(image_file.read()).decode("utf-8")

def create_dummy_image(filepath):
    if not os.path.exists(filepath):
        img = Image.new('RGB', (400, 200), color=(100, 149, 237))
        d = ImageDraw.Draw(img)
        d.text((20, 80), "LangChain 1.0 Agent", fill=(255, 255, 255))
        img.save(filepath)

# [Tool] 파일 저장 도구
@tool
def save_file_tool(content: str) -> str:
    """
    분석된 텍스트 내용을 파일(report.txt)에 저장합니다.
    Args:
        content: 저장할 텍스트 내용
    """
    try:
        # 파이썬에서 file I/O
        # with는 파이썬 스코프
        # f라는 이름의 file handler 통해서 파일 내용물 f.write 쓸 수 있음
        # 스코프 왜 쓰나? 스코프가 종료되면서 file handler로 del됨.
        # 스코프 안 쓰면 f.close()로 직접 닫을 수 있음.
        with open(f"report{MODEL_NAME}.txt", "w", encoding="utf-8") as f:
            f.write(content)
        return "✅ 파일 저장 완료: report.txt"
    except Exception as e:
        return f"❌ 저장 실패: {e}"

### 🤖 4. LangChain 멀티모달 에이전트 생성 및 실행
* `ChatOllama`를 통해 비전 모델을 초기화하고, 도구(Tool)를 바인딩하여 에이전트를 생성합니다.
* 사용자의 텍스트 요청과 이미지를 결합하여 모델에 전달하고 결과를 받아옵니다.

In [5]:
# ==========================================
# 3. LangChain 1.0 Agent 생성
# ==========================================
def run_modern_agent(image_path):
    # (1) 모델 초기화
    llm = ChatOllama(model=MODEL_NAME, temperature=0)
    # base_url?
    # ChatOllama 기본값으로 들어가있음. localhost:11434
    # api_key?
    # 로컬로 쓰다 보니 불필요.

    # (2) 도구 리스트
    tools = [save_file_tool]

    # (3) 에이전트 생성
    system_prompt = (
        "당신은 이미지를 분석하고 결과를 파일로 저장하는 유능한 AI 비서입니다. "
        "사용자의 요청에 따라 이미지를 분석한 뒤, 반드시 'save_file_tool'을 사용하여 결과를 저장하세요."
    )

    agent = create_agent(
        model=llm,
        tools=tools,
        system_prompt=system_prompt,
    )

    # (4) 실행 (멀티모달 메시지 구성)
    # 이미지 전처리
    b64_img = encode_image(image_path)

    # 메시지 리스트 구성
    messages = [
        HumanMessage(
            content=[
                {
                    "type": "text",
                    "text": "이 이미지를 자세히 분석해서 한국어로 요약해줘. 그리고 분석 내용을 파일로 저장해."
                },
                {
                    "type": "image_url",
                    "image_url": {"url": f"data:image/jpeg;base64,{b64_img}"}
                }
            ]
        )
    ]

    print("\n🤖 LangChain 1.0 에이전트 실행 중...")

    # invoke 실행
    result = agent.invoke({"messages": messages})

    # 결과 추출 (마지막 메시지가 AI의 최종 응답)
    return result["messages"][-1].content

### ▶️ 6. 메인 실행 및 결과 확인
* 서버 시작부터 이미지 생성, 에이전트 실행까지의 전체 흐름을 주관합니다.
* 실행 후 `report.txt` 파일이 정상적으로 생성되었는지 내용을 출력하여 검증합니다.

In [7]:
# ==========================================
# 메인 실행
# ==========================================
if __name__ == "__main__":
    start_ollama_server()
    # create_dummy_image(IMAGE_PATH)

    # 실행
    final_response = run_modern_agent(IMAGE_PATH)

    print("\n" + "="*30 + " 최종 응답 " + "="*30)
    print(final_response)

    # 파일 확인
    fn = f"report{MODEL_NAME}.txt"
    if os.path.exists(fn):
        with open(fn, "r") as f:
            print("\n📂 report 내용 확인:")
            print(f.read())

✅ 모델 준비됨: qwen3.5:4b

🤖 LangChain 1.0 에이전트 실행 중...

============================== 최종 응답 ==============================
✅ 이미지 분석 및 파일 저장 완료!

**분석 요약:**
갈색과 검은색 무늬의 타비 고양이가 창틀 위의 담요 위에 편안하게 누워 있습니다. 자연광이 들어오는 밝고 따뜻한 분위기 속에서, 고양이는 창밖을 바라보며 평화로운 모습을 하고 있습니다. 배경에는 다육식물과 책들이 놓여 있어 안락한 실내 공간임을 알 수 있습니다.

분석 내용은 `report.txt` 파일로 저장되었습니다.

📂 report 내용 확인:
이미지 분석 요약:

*   **주체:** 갈색과 검은색 무늬가 있는 고양이 (타비 고양이의 특징) 가 흰 발가락을 보이며 누워 있습니다.
*   **위치 및 환경:** 나무로 된 창틀 위에 놓인 회색/흰색의 얇은 베개나 담요 위에 편안하게 쉬고 있습니다.
*   **배경:** 왼쪽에는 큰 창문이 있고, 밖으로는 초록색 식물과 자연광이 비치고 있습니다. 고양이의 뒤쪽에는 작은 다육식물이 있는 테라코타 화분이 보입니다. 오른쪽 배경에는 책상이나 서랍장이 보이며, 그 위에 책들이 놓여 있습니다.
*   **분위기:** 창문에서 들어오는 자연광으로 밝고 따뜻한 분위기를 연출하며, 고양이는 창밖을 바라보며 평화롭고 여유로운 모습을 하고 있습니다.


### 💡 Fallback: 이미지 분석까지만 해보기
* 파일 저장 단계에서 모델이 오류를 내는 중인가요?
* 그럼 그 전 단계, 즉 이미지 분석까지는 잘 하는지 아래 코드로 확인해보세요.
* `requests` 라이브러리를 사용하여 이미지를 가져오고, 이를 Base64 포맷으로 변환하여 모델에게 설명을 요청합니다.

In [8]:
import base64
import os
import requests
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage

def analyze_image(image_source):
    """
    로컬 경로 또는 웹 URL에서 이미지를 읽어 LLM에게 분석을 요청합니다.
    """
    print(f"📥 이미지 읽는 중: {image_source}")
    try:
        if image_source.startswith("http://") or image_source.startswith("https://"):
            img_data = requests.get(image_source).content
        else:
            if not os.path.exists(image_source):
                return f"❌ 이미지 파일이 존재하지 않습니다: {image_source}"
            with open(image_source, "rb") as f:
                img_data = f.read()
        b64_img = base64.b64encode(img_data).decode("utf-8")
    except Exception as e:
        return f"❌ 오류 발생: {e}"

    try:
        llm = ChatOllama(model=MODEL_NAME, temperature=0)
    except NameError:
        return "❌ 오류: MODEL_NAME 변수가 정의되지 않았습니다."

    messages = [
        HumanMessage(
            content=[
                {"type": "text", "text": "이 이미지는 어떤 내용인가요? 자세히 묘사해 주세요."},
                {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{b64_img}"}}
            ]
        )
    ]
    print("🤖 분석 실행 중...")
    result = llm.invoke(messages)
    return result.content

# 로컬
# analysis_result = analyze_image("/home/user/MultiAgents/Day_2/Ex/image_data/cat.jpg")

# 웹 URL
analysis_result = analyze_image(IMAGE_PATH)

print(f"기저 모델: {MODEL_NAME}")
print("\n" + "="*30 + " 분석 결과 " + "="*30)
print(analysis_result)

📥 이미지 읽는 중: /home/user/MultiAgents/Day_2/Ex/image_data/cat.jpg
🤖 분석 실행 중...
기저 모델: qwen3.5:4b

============================== 분석 결과 ==============================
이 이미지는 창가에서 휴식을 취하고 있는 평화로운 분위기를 풍기는 고양이 한 마리를 담고 있습니다.以下是详细的描述：

**1. 주체 (고양이):**
*   **외모:** 이미지 중앙에 누워있는 고양이는 갈색과 검은색 줄무늬가 섞인 타비(Tabby) 외모를 하고 있으며, 가슴과 앞발은 흰색입니다.
*   **자세:** 고양이는 몸을 살짝 말려서 편안하게 누워 있고, 머리는 앞발 위에 얹어 resting 하고 있습니다.
*   **표정:** 고양이 카메라를 직시하고 있으며, 노란색 눈으로 고요하고 차분한 표정을 짓고 있습니다.

**2. 주변 환경 (이탈리아):**
*   **침대/담요:** 고양이는 회색과 흰색 줄무늬가 있는 얇은 니트 담요 위에 누워 있습니다. 담요의 질감이 부드럽고 따뜻해 보입니다.
*   **창문:** 왼쪽에는 큰 창문이 보이며, 자연광이 실내로 들어와 고양이와 주변 물체들을 밝게 비추고 있습니다.
*   **식물:** 고양이의 뒤쪽에는 terracotta(타르코타) 화분에 심어진 작은 수생식물 (수선화나 알로에 같은 succulent) 이 놓여 있습니다. 창문 밖으로는 흐릿하게 보이는 녹색 식물들이 있어 자연스러운 느낌을 줍니다.

**3. 배경:**
*   **책장:** 이미지 오른쪽 배경에는 나무 책장이 보이며, 그 위에는 책들과 woven 바구니가 쌓여 있습니다.
*   **바닥/대기:** 바닥은 목재로 되어 있고, 전체적으로 햇살이 들어오는 오후나 아침 시간대의 따뜻한 조명이 감도는 듯한 분위기입니다.

전체적으로 이 이미지는 집 안의 한 구석에서 편안하게 쉬고 있는 고양이의 일상적인 순간을 포착한 듯한, 매우 차분하고 아름다운 장면입니다.
